In [37]:
from langchain.document_loaders import TextLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.embeddings import OllamaEmbeddings
from langchain.vectorstores.pgvector import PGVector
import psycopg2
from tqdm import tqdm
import time

In [40]:
loader = TextLoader("indian_constitution.txt", encoding="utf-8")
docs = loader.load()
splitter = RecursiveCharacterTextSplitter(
    chunk_size=512,
    chunk_overlap=50
)
chunks = splitter.split_documents(docs)

In [44]:
# --- Embed chunks with timer ---
print(f"🧠 Embedding {len(chunks)} chunks with 'nomic-embed-text'...")
start_embed = time.time()

embeddings = OllamaEmbeddings(model="nomic-embed-text")
embedding_list = [
    embeddings.embed_query(chunk.page_content) for chunk in tqdm(chunks, desc="🔄 Embedding")
]

embed_duration = time.time() - start_embed
print(f"✅ Embedding done in {embed_duration:.2f} seconds ({embed_duration / len(embedding_list):.2f} sec/chunk)\n")

# --- Insert into pgvector with timer ---
if len(embedding_list) == 0:
    print("⚠️ No embeddings to insert.")
else:
    print(f"📥 Inserting {len(embedding_list)} vectors into pgvector...")
    conn = psycopg2.connect("dbname=constdb user=postgres password=5107")
    cur = conn.cursor()

    start_insert = time.time()

    for i in tqdm(range(len(embedding_list)), desc="📌 Inserting into DB"):
        content = chunks[i].page_content
        embedding = embedding_list[i]

        cur.execute(
            "INSERT INTO vectordb (content, embedding) VALUES (%s, %s)",
            (content, embedding)
        )

    conn.commit()
    cur.close()
    conn.close()

    insert_duration = time.time() - start_insert
    print(f"✅ Insert done in {insert_duration:.2f} seconds ({insert_duration / len(embedding_list):.2f} sec/row)")

🧠 Embedding 2247 chunks with 'nomic-embed-text'...


🔄 Embedding: 100%|██████████| 2247/2247 [1:20:09<00:00,  2.14s/it]


✅ Embedding done in 4810.04 seconds (2.14 sec/chunk)

📥 Inserting 2247 vectors into pgvector...


📌 Inserting into DB: 100%|██████████| 2247/2247 [00:03<00:00, 583.96it/s]

✅ Insert done in 3.85 seconds (0.00 sec/row)
